In [14]:
import jax
import jax.numpy as jnp
import agent
import env
import matplotlib.pyplot as plt
import numpy as np

# K-armed bandit functions
try every lever initialization, ucb bandit, and run single bandit. 

run_single_bandit_seed combines pick_every_lever_innit and UCB_badit to create a single function with both. 

In [ ]:
#Initializing true means and exploration
true_mu = jnp.array([-1.0, -0.5, 0.0, 0.3, 0.6, 0.9, 1.1, 1.0, 0.4, -0.8])
c = 0.5

def pick_every_lever_innit(state, filler):
     policy = agent.pick_every_lever(state)
     state,n = env.k_slots(policy, state, true_mu)
     return state,n


def UCB_bandit(state, filler):
     policy = agent.UCB_agent(state,c)
     state, n = env.k_slots(policy, state, true_mu)
     return state, n


def run_single_bandit_seed(key):
     t = 0
     state = (t, jnp.zeros(len(true_mu)), jnp.zeros(len(true_mu)), key)
     state, n = jax.lax.scan(pick_every_lever_innit,state, length=len(true_mu))
     state, n = jax.lax.scan(UCB_bandit, state, length=10000)
     return n[-1]






# Test
Test different bandits

In [ ]:

#creating keys
num_seeds = 100
master_key = jax.random.key(0)
keys = jax.random.split(master_key, num_seeds)

#Creating vmap function so we can parralelize 
run_parallel_seeds = jax.vmap(run_single_bandit_seed)
# Using vmaped function to test all 100 seeds at once
n_hist_100_seeds = run_parallel_seeds(keys)

#Getting mean and std
mean_counts = jnp.average(n_hist_100_seeds, axis = 0)
std_counts = jnp.std(n_hist_100_seeds, axis = 0)

print(mean_counts)
print(std_counts)

[1.2900000e+00 2.0599999e+00 2.8099999e+00 5.4099998e+00 9.3499994e+00
 3.1844000e+02 8.0455098e+03 1.6173099e+03 6.3499999e+00 1.4699999e+00]
[5.3469604e-01 1.5021316e+00 2.4726300e+00 5.0657578e+00 1.1212828e+01
 1.4281693e+03 3.5684675e+03 3.3241460e+03 6.5534339e+00 6.8490875e-01]
